<style>
.topic-header { background: linear-gradient(135deg, #e8f4f8 0%, #d4e8f0 100%); border-left: 4px solid #5ba4c9; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0; font-size: 15px; color: #1a3a4a; }
.concept-box { background: #eef6fa; border: 1px solid #c4dce8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a4a5a; }
.try-it { background: #fef9e7; border: 1px solid #f0d87a; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #5a4a1a; }
.warning-box { background: #fdf0f0; border: 1px solid #e8b0b0; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #6a2a2a; }
.takeaway { background: #e8f5e8; border: 1px solid #a8d5a8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a5a2a; }
.where-box { background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #4a3000; }
.fix-box { background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #1b5e20; }
.section-divider { border: none; border-top: 2px solid #d4e8f0; margin: 25px 0; }
</style>

<div style="background: linear-gradient(135deg, #e8f4f8 0%, #c4dce8 100%); padding: 30px 32px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: #1a3a4a; margin: 0; font-size: 28px;">E09 &middot; Day 3 &middot; Plain-English Analytics &mdash; SQL-Database RAG</h1>
<p style="color: #3a6a8a; margin: 8px 0 0 0; font-size: 16px;">GenAI for Engineering Managers &mdash; Exercise 9 of 15 &middot; Facilitator-run (watch, or try alongside)</p>
<p style="color: #2a4a5a; margin: 14px 0 0 0; font-size: 14px; line-height: 1.6;">
<strong>Why this exercise:</strong> In E08 we grounded the assistant in documents &mdash; policies, runbooks, seller-help articles. But that closed only half the gap. The questions your leadership actually asks &mdash; "which region's inventory turns dropped last quarter?", "how many POS outages did we log in June?" &mdash; are not answered by any document. They live in <strong>structured data</strong>: the sales, inventory, and incident databases your teams operate. Today the answer path is a ticket to an analyst and a two-day wait. In this exercise the LLM writes the SQL itself: plain-English question in, governed query out, business answer back. And &mdash; because you will be asked to approve exactly this kind of system &mdash; we spend real time on the failure mode that matters most: the query that runs perfectly, returns a confident number, <em>and is wrong</em>.
</p>
</div>


<div class="topic-header">
<strong>Part 0 &mdash; Setup: our retail operations database</strong>
</div>

<div class="concept-box">
One script builds <code>retail_ops.db</code> &mdash; a realistic slice of retail operations data: 4 regions, 24 stores, and 60 weeks (Jul&nbsp;2025 &ndash; Aug&nbsp;2026) of weekly sales and inventory across 5 categories, plus an operational incident log. About 14,000 rows &mdash; small enough for a notebook, rich enough that the questions get interesting.
</div>

In [ ]:
%pip install -q openai

In [ ]:
import os, sqlite3
import openai

# Build (or rebuild) the database
%run ../data/setup_retail_data.py

DB_PATH = os.path.join('..', 'data', 'retail_ops.db')
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

client = openai.OpenAI(
    api_key='PASTE_THE_KEY_SHARED_IN_SESSION_HERE'
)
MODEL = 'gpt-5.4-nano'
print('\nDatabase connected. Model:', MODEL)

<hr class="section-divider">

<div class="topic-header">
<strong>Part 1 &mdash; The barrier: every business question is a SQL ticket</strong>
</div>

<div class="where-box">
<strong>WHERE it hurts today:</strong> A regional VP asks "which region is struggling on inventory turns?" That question becomes a Jira ticket, waits in an analyst queue behind twelve others, and comes back in two days as a spreadsheet. The data was sitting in a database the whole time &mdash; the only barrier was that the VP does not write SQL.
</div>

Here is what the analyst would write. Look at it the way your stakeholders do &mdash; as a foreign language:

In [ ]:
# The analyst's answer to "how are inventory turns trending by region this quarter vs last?"
# (annualised approximation: quarterly units sold / average weekly on-hand)
sql = '''
SELECT r.region_name,
       ROUND(SUM(CASE WHEN s.week_start BETWEEN '2026-01-01' AND '2026-03-31'
                      THEN s.units_sold END) * 1.0
             / AVG(CASE WHEN i.week_start BETWEEN '2026-01-01' AND '2026-03-31'
                        THEN i.units_on_hand END), 1)  AS q1_2026_turns,
       ROUND(SUM(CASE WHEN s.week_start BETWEEN '2026-04-01' AND '2026-06-30'
                      THEN s.units_sold END) * 1.0
             / AVG(CASE WHEN i.week_start BETWEEN '2026-04-01' AND '2026-06-30'
                        THEN i.units_on_hand END), 1)  AS q2_2026_turns
FROM sales s
JOIN inventory i ON i.store_id = s.store_id
                AND i.week_start = s.week_start
                AND i.category  = s.category
JOIN stores  st ON st.store_id = s.store_id
JOIN regions r  ON r.region_id = st.region_id
GROUP BY r.region_name
ORDER BY q2_2026_turns
'''
print(f"{'Region':<12} {'Q1-2026':>10} {'Q2-2026':>10}")
print('-' * 34)
for region, q1, q2 in cursor.execute(sql):
    flag = '   <-- something happened here' if q2 < q1 * 0.7 else ''
    print(f"{region:<12} {q1:>10} {q2:>10}{flag}")
print('\nFour JOINs, two CASE windows, a ratio metric. This is why the ticket queue exists.')

<div class="concept-box">
The data just told us a real story &mdash; <strong>Southeast inventory turns roughly halved in Q2 2026</strong> &mdash; but only because someone who speaks SQL went looking. The idea of SQL-database RAG: instead of retrieving <em>documents</em> (E08), we give the model the <strong>database schema</strong> as its context, let it <em>write</em> the query, execute the query ourselves, and hand the rows back to the model to narrate. Retrieval-augmented, but the "retrieval" is a governed SQL execution.
</div>

<hr class="section-divider">

<div class="topic-header">
<strong>Part 2 &mdash; Plain English in, SQL out</strong>
</div>

<div class="concept-box">
The pipeline your teams would build has four steps &mdash; and the division of labour matters for governance:<br><br>
<strong>1. Schema as context</strong> &mdash; the model is handed table/column definitions (not the data).<br>
<strong>2. Model writes SQL</strong> &mdash; text generation, nothing executed yet.<br>
<strong>3. Your code executes it</strong> &mdash; on a read-only connection, inside your perimeter. The model never touches the database.<br>
<strong>4. Model narrates the rows</strong> &mdash; turns the result set into a business answer.
</div>

In [ ]:
DB_SCHEMA = '''
Tables in retail_ops.db (SQLite):

regions(region_id, region_name)
stores(store_id, store_name, region_id, store_format, city, opened_date)
sales(sale_id, store_id, week_start, category, units_sold, revenue)
inventory(inv_id, store_id, week_start, category, units_on_hand)
incidents(incident_id, store, incident_type, severity, opened_date, status, description)

Notes:
- regions: Northeast, Southeast, Midwest, West
- store_format: Supercenter, Neighborhood Market, Discount Store
- categories: Grocery, Electronics, Apparel, Home & Garden, Pharmacy
- sales and inventory are WEEKLY per store per category; week_start is YYYY-MM-DD
- data covers weeks from 2025-07-07 through 2026-08-10
- incidents.severity: low/medium/high; incidents.status: open/resolved
- revenue is in US dollars
'''

def nl_to_sql(question, schema=DB_SCHEMA, extra_rules=''):
    prompt = f'''You are a SQL analyst. Write ONE SQLite query answering the question.

{schema}

Rules:
- Return ONLY the SQL query, no explanation, no markdown fences.
- Use proper JOINs where needed.
{extra_rules}
Question: {question}

SQL:'''
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
    )
    sql = resp.choices[0].message.content.strip()
    return sql.replace('```sql', '').replace('```', '').strip()

def run_sql(sql, limit=10):
    try:
        cursor.execute(sql)
        cols = [d[0] for d in cursor.description]
        rows = cursor.fetchall()
        return {'sql': sql, 'columns': cols, 'rows': rows, 'error': None}
    except Exception as e:
        return {'sql': sql, 'columns': [], 'rows': [], 'error': str(e)}

def ask_database(question, **kw):
    result = run_sql(nl_to_sql(question, **kw))
    print(f"Q: {question}")
    print(f"Generated SQL:\n{result['sql']}\n")
    if result['error']:
        print(f"SQL ERROR: {result['error']}")
    else:
        print(' | '.join(result['columns']))
        for row in result['rows'][:10]:
            print('  ', row)
        if len(result['rows']) > 10:
            print(f"   ... ({len(result['rows'])} rows total)")
    print('=' * 70)
    return result

_ = ask_database('How many stores do we have in each region, broken down by store format?')

In [ ]:
# Three questions of increasing difficulty -- the last one needs a
# three-table JOIN plus a ratio metric the model must infer.
r1 = ask_database('What were our top 3 categories by total revenue in July 2026?')
r2 = ask_database('Which 5 stores sold the most Apparel units in 2026?')
r3 = ask_database("Which region's inventory turns dropped in Q2 2026 compared to Q1 2026? "
                  'Compute turns as total units sold divided by average units on hand per quarter. '
                  'Show every region with its Q1 turns, Q2 turns, and the change, ordered by the change.')

<div class="try-it">
<strong>Pause and look at what just happened.</strong> The third question is the one our VP asked. The model reconstructed &mdash; from a plain-English sentence and a schema &mdash; a multi-JOIN, two-window query of the same shape the analyst wrote in Part&nbsp;1, and the Southeast drop shows up in the result. No ticket, no queue. That is the demo your teams will show you. Notice, though, how much the phrasing mattered: we had to ask for "every region with its Q1 turns, Q2 turns, and the change" to get a readable answer &mdash; a vaguer phrasing of the same question returns a bare list of region names with no magnitudes. <em>Hold the applause until Part&nbsp;4.</em>
</div>

<hr class="section-divider">

<div class="topic-header">
<strong>Part 3 &mdash; From rows to a business answer</strong>
</div>

<div class="concept-box">
Raw rows are still analyst-speak. The last step hands the SQL, the columns, and the rows back to the model and asks for a stakeholder-ready answer. This is the part your leadership actually sees &mdash; which is exactly why the query underneath must be trustworthy.
</div>

In [ ]:
def ask_and_answer(question, **kw):
    # Production pattern: if the generated SQL errors, retry ONCE feeding the
    # error message back to the model. (Loud failures are recoverable --
    # Part 4 shows the failures that are not.)
    result = run_sql(nl_to_sql(question, **kw))
    if result['error']:
        print(f"  [retry: first query errored -> {result['error']}]")
        result = run_sql(nl_to_sql(
            question + '\nNote: a previous attempt failed with this SQLite error, '
                       f"avoid it: {result['error']}\nFailed SQL: {result['sql']}", **kw))
    if result['error']:
        return f"Query failed twice: {result['error']}"
    synthesis = f'''Answer the user's question from this database query result.

Question: {question}
SQL used: {result['sql']}
Columns: {result['columns']}
Rows: {result['rows'][:25]}

Give a concise, business-friendly answer with the key numbers, then one sentence
on any notable pattern. Do not mention SQL.'''
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': synthesis}],
    )
    return resp.choices[0].message.content

for q in [
    "Which region's inventory turns dropped in Q2 2026 vs Q1 2026, and by how much? "
    'Compute turns as total units sold divided by average units on hand per quarter.',
    'Were inventory system sync errors more common in Southeast stores than elsewhere '
    'between April and June 2026? incidents.store contains the store name.',
]:
    print(f'Q: {q}\n')
    print(f'A: {ask_and_answer(q)}')
    print('=' * 70)

<hr class="section-divider">

<div class="topic-header">
<strong>Part 4 &mdash; The honest demo: a confident, wrong answer</strong>
</div>

<div class="warning-box">
<strong>The failure mode that should keep you up at night is not the query that errors.</strong> Errors are loud; someone fixes them. The dangerous query is the one that <em>runs cleanly, returns a plausible number, and is wrong</em> &mdash; because a wrong number delivered fluently to a VP becomes a decision. Below we ask a question a manager would genuinely ask, with two traps a real warehouse has:<br><br>
<strong>Trap 1 &mdash; "last quarter" is ambiguous.</strong> The model does not know today's date. Last <em>calendar</em> quarter (Q2 2026)? The last 90 days? The model will silently pick one.<br>
<strong>Trap 2 &mdash; a legacy join key.</strong> Our <code>incidents</code> table comes from an old ticketing system: its <code>store</code> column holds the store <em>name</em> ("Store 204 - Nashville West"), not the numeric <code>store_id</code>. The schema listing alone does not say so &mdash; exactly like the schema dump an engineer would paste into a prompt.
</div>

In [ ]:
danger_question = ('How many incidents did our Southeast region stores have last quarter, '
                   'and how does that compare to the other regions?')

danger = ask_database(danger_question)

# Ground truth, written by hand with full knowledge of the data:
# today is 2026-08-14, so 'last quarter' (calendar) = Q2 2026,
# and incidents.store must be matched to stores.store_name.
truth_sql = '''
SELECT r.region_name, COUNT(*) AS incidents_q2_2026
FROM incidents i
JOIN stores  st ON st.store_name = i.store
JOIN regions r  ON r.region_id   = st.region_id
WHERE i.opened_date BETWEEN '2026-04-01' AND '2026-06-30'
GROUP BY r.region_name ORDER BY incidents_q2_2026 DESC
'''
print('GROUND TRUTH (hand-written, Q2 2026, joined on store NAME):')
for row in cursor.execute(truth_sql):
    print('  ', row)

<div class="warning-box">
<strong>Dissect what the model did.</strong> Compare its query to the ground truth above, line by line, and check two things:<br><br>
<strong>1. The join.</strong> Did it write <code>ON i.store = st.store_id</code> &mdash; matching a text name against a numeric ID? In SQLite that comparison quietly matches nothing (or nonsense): the query <em>succeeds</em> and returns zeros or an empty set. No error, no warning. In a dashboard this reads as "no incidents &mdash; great quarter!"<br><br>
<strong>2. The date window.</strong> What did it decide "last quarter" means? Whatever it chose, <em>you</em> did not choose it &mdash; and a different phrasing tomorrow may silently choose differently, so two executives can get two different "last quarters" from the same system.<br><br>
Either failure alone produces a clean-running query with a wrong number. Run the cell above a few times &mdash; the model may guess right on one trap and wrong on the other, and <strong>the output looks equally confident either way</strong>. That variance is itself the lesson: correctness by coin-flip is not correctness.
</div>

In [ ]:
# Make the failure measurable: compare the model's answer for Southeast
# against ground truth.
truth = {r[0]: r[1] for r in cursor.execute(truth_sql)}
print(f"Ground truth, Q2 2026:  {truth}")

if danger['error']:
    print(f"\nModel query ERRORED: {danger['error']}")
    print('An error is the GOOD outcome -- someone will notice and fix it.')
else:
    print(f"\nModel query returned {len(danger['rows'])} row(s): {danger['rows'][:6]}")
    se_values = [v for row in danger['rows'] if any('southeast' in str(x).lower() for x in row)
                 for v in row if isinstance(v, (int, float))]
    if not danger['rows'] or (se_values and se_values[-1] != truth.get('Southeast')):
        print(f"\n*** WRONG, SILENTLY. Southeast truth = {truth.get('Southeast')}, "
              f"model's number = {se_values[-1] if se_values else 'nothing at all'}. ***")
    elif se_values:
        print('\nThis run it happened to match ground truth -- re-run and watch it wobble.')

<div class="fix-box">
<strong>The corrected pattern &mdash; what disciplined teams do differently.</strong> None of these fixes are exotic; they are engineering-management decisions, not research problems:<br><br>
<strong>1. Semantic schema, not raw schema.</strong> The prompt must carry the <em>meaning</em>: "<code>incidents.store</code> holds the store <em>name</em>; join it to <code>stores.store_name</code>." A raw <code>DESCRIBE</code> dump is not enough context.<br>
<strong>2. Resolve time words before the model sees them.</strong> "Last quarter" gets translated to explicit dates <em>by your code</em> (calendar-aware, timezone-aware, fiscal-calendar-aware), never left to the model's imagination.<br>
<strong>3. A curated semantic layer beats raw tables.</strong> Production NL-to-SQL systems query governed views with business definitions baked in ("inventory turns" defined once, by finance) &mdash; the model picks views, it does not invent metrics.<br>
<strong>4. Show the SQL.</strong> Every answer carries its query, so an analyst can audit in ten seconds what took a VP ten days to mis-decide.
</div>

In [ ]:
from datetime import date

# Fixes 1 + 2: semantic schema notes, and 'last quarter' resolved by CODE.
SEMANTIC_NOTES = '''
CRITICAL join semantics:
- incidents.store contains the store NAME (e.g. "Store 204 - Nashville West").
  To join incidents to stores you MUST use: incidents.store = stores.store_name
  (NEVER join incidents.store to stores.store_id).
'''

def resolve_last_quarter(today=date(2026, 8, 14)):
    q = (today.month - 1) // 3           # current quarter index 0-3
    year, prev_q = (today.year, q - 1) if q > 0 else (today.year - 1, 3)
    start_month = prev_q * 3 + 1
    end_month = start_month + 2
    end_day = {3: 31, 6: 30, 9: 30, 12: 31}[end_month]
    return (date(year, start_month, 1).isoformat(),
            date(year, end_month, end_day).isoformat())

start, end = resolve_last_quarter()
print(f"'Last quarter' resolved by code -> {start} to {end}\n")

fixed_question = (f'How many incidents did stores in each region have between {start} '
                  f'and {end}? Order by count descending.')

fixed = ask_database(fixed_question, schema=DB_SCHEMA + SEMANTIC_NOTES)

match = {r[0]: r[1] for r in cursor.execute(truth_sql)}
got = {str(row[0]): row[-1] for row in fixed['rows']} if not fixed['error'] else {}
print('Matches ground truth:', got == match, '| truth =', match)

<hr class="section-divider">

<div class="topic-header">
<strong>Part 5 &mdash; Guardrails: the model proposes, your code disposes</strong>
</div>

<div class="warning-box">
One more governance question before you approve this system: <strong>what stops the model from generating <code>DROP TABLE</code>?</strong> Nothing &mdash; unless your code checks. A user can type "delete all the incidents from January" in perfectly polite English, and the model will often write perfectly valid destructive SQL &mdash; and even when it happens to decline (models sometimes answer a "drop the table" request with a harmless SELECT instead), you cannot bet the warehouse on the model's mood. Two layers, both cheap and deterministic: a read-only database connection (the real enforcement), and a SQL validator in front of execution (fast feedback). Watch below which requests the model complies with &mdash; and notice the validator does not care either way.
</div>

In [ ]:
def safe_ask_database(question, **kw):
    sql = nl_to_sql(question, **kw)
    sql_upper = sql.upper()
    for keyword in ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE',
                    'CREATE', 'PRAGMA', 'ATTACH']:
        if keyword in sql_upper:
            return {'blocked': True, 'reason': f"generated SQL contains '{keyword}'", 'sql': sql}
    if not sql_upper.lstrip().startswith(('SELECT', 'WITH')):
        return {'blocked': True, 'reason': 'only SELECT queries are allowed', 'sql': sql}
    return {'blocked': False, **run_sql(sql)}

for q in ['Which store format sells the most Electronics units?',
          'Delete all incidents opened before 2026',
          'Drop the inventory table, we are rebuilding it']:
    r = safe_ask_database(q)
    verdict = f"BLOCKED ({r['reason']})" if r['blocked'] else 'ALLOWED'
    print(f'{verdict:<55} <- "{q}"')
    print(f'   generated SQL was: {r["sql"][:80]}...' if len(r['sql']) > 80
          else f'   generated SQL was: {r["sql"]}')
    print()

conn.close()
print('Database connection closed.')

<hr class="section-divider">

<div class="takeaway">
<strong>Key Takeaways for Engineering Managers</strong>
<ol>
<li><strong>SQL-database RAG closes E08's other half</strong> &mdash; documents ground the words, databases ground the numbers. The "retrieval" here is a governed query execution, with the schema as the context.</li>
<li><strong>The model writes SQL; it never touches the database.</strong> Your code executes, on a read-only connection, inside your perimeter. That division of labour is the whole security story &mdash; insist on it in design reviews.</li>
<li><strong>The scary failure is silent, not loud.</strong> A join between a name and an ID, or a guessed "last quarter", runs cleanly and returns a confident wrong number. Ask every NL-to-SQL demo: <em>"show me a question it gets subtly wrong."</em></li>
<li><strong>The fixes are managerial, not magical:</strong> semantic schema notes, time words resolved by code, a curated metric layer, and the SQL shown with every answer.</li>
<li><strong>Guardrails are two cheap layers</strong> &mdash; a read-only connection plus a SELECT-only validator. If a vendor's demo lacks either, that is your first question.</li>
<li><strong>The business case is the ticket queue:</strong> every question answered here in seconds is today a two-day analyst round-trip. Measure the pilot in tickets avoided.</li>
</ol>
</div>

<div class='topic-header'>
<strong>&#128279; The gap we leave &mdash; and where E10 begins</strong><br><br>
Our assistant can now read documents (E08) and interrogate databases (E09) &mdash; it can <em>answer</em> almost anything about the business. But notice what it still cannot do: <strong>act</strong>. It found the Southeast inventory problem; it cannot open the replenishment ticket, page the on-call, check the live order-management system, or reach any tool outside this notebook. And every integration we built here was hand-wired &mdash; a bespoke <code>run_sql()</code> for this one database. Wiring N models to M tools that way does not scale past a demo.<br><br>
<strong>E10</strong> fixes exactly that with <strong>tool calling and MCP (Model Context Protocol)</strong>: a standard way for the model to discover live systems, call them safely, and act &mdash; with your code still holding the keys.
</div>